# Notebook 01 — pandas Fundamentals with Market Data

**FIN 4600 · Lab 3 · Financial Data Analytics**

Duran, *Financial Services Technology* (3rd ed.), **Chapter 6 — Data Analytics**

---

Chapter 6 makes the point that analytics is mostly *data management*: the
modelling is the last and smallest step. This notebook is the data-management
step. By the end of it you will be able to load a raw market data file,
inspect it, reshape it, aggregate it, join reference data onto it, and handle
the gaps — which is roughly 80% of what a financial data analyst does.

**What you will learn**

| Skill | Why a finance professional needs it |
|---|---|
| `read_csv`, dtypes, dates | Vendor files arrive as text; a date stored as text cannot be sorted or differenced |
| `.loc` / `.iloc` / boolean masks | Selecting the rows a risk report actually covers |
| `groupby` | Per-desk, per-instrument, per-sector aggregation |
| `pivot` (long ↔ wide) | Market data arrives long; portfolio maths needs wide |
| `resample` | Daily marks → month-end reporting periods |
| `merge` | Attaching reference/security-master data to transactions |
| Missing-data handling | Holidays, halts, delistings, late feeds |

**Datasets used**

- `data/sp500_prices.csv` — real daily OHLCV for 30 large-cap US stocks,
  Feb 2013 – Feb 2018
- `data/sp500_companies.csv` — a security master: ticker, company name, GICS sector

**How to use this notebook.** All the code is provided, including worked
solutions to the three exercises. Run every cell in order (`Shift + Enter`),
read the comments in the code and the explanation above each cell, and check
the output against what the text tells you to expect. A Canvas comprehension
quiz on this notebook follows.

## 0. Setup

Every notebook starts with this same block. It finds the repository, points at
the `data` folder, and loads the shared chart style.

#### Background: the three libraries you meet in this notebook

**NumPy** (`import numpy as np`) is the numerical engine underneath almost
all scientific Python. Its core object is the `ndarray`: a block of numbers
that all share one type (say 64-bit floats), stored side by side in memory.
Because the layout is fixed, NumPy can run maths over a million numbers in
compiled C code instead of one Python step at a time. Every pandas column is,
underneath, a NumPy array.

**pandas** (`import pandas as pd`) adds *labels* to NumPy arrays. Wes McKinney
wrote it in 2008 while working at the hedge fund AQR, because the tools of the
day could not handle dated, labelled market data well. It is now the default
data tool in banks, asset managers, insurers and fintechs. Its two objects:

- a **Series** is one column: an array of values plus an **index**, the row
  labels (row numbers, dates, tickers...);
- a **DataFrame** is a table: several Series that share the same index, each
  with a column name. Each column has a single **dtype** (data type).

Almost every pandas operation lines data up by index label, not by position.
That *alignment* is what makes it safe to combine two price series that start
on different days.

**matplotlib** (`import matplotlib.pyplot as plt`) is the standard Python
charting library, modelled on MATLAB's plotting. It gives fine control over
every element of a chart, which is why regulatory and client reports are
often drawn with it. `plt.style.use(...)` loads a style file so every chart
in the course has the same look.

The short names `np`, `pd` and `plt` are universal conventions; you will see
them in every textbook and codebase.

**What the next cell does.** Imports the three libraries, defines the same
`find_repo_root` helper you met in Notebook 00 (it climbs up the folder tree
until it finds `data/sp500_prices.csv`), loads the chart style, and widens
pandas' display so wide tables are not cut off. `pd.set_option` changes only
how tables are *printed*, never the data.

**What to look for:** one line printing the repository root folder.

In [ ]:
# pathlib.Path represents file and folder locations as objects that can be joined with "/"
from pathlib import Path

# NumPy: fast numeric arrays; "np" is the universal short name
import numpy as np
# pandas: labelled tables (DataFrame) and columns (Series); "pd" is the universal short name
import pandas as pd
# pyplot: matplotlib's chart-drawing interface; "plt" is the universal short name
import matplotlib.pyplot as plt


# Same helper as Notebook 00: find the project folder wherever the notebook was started from
def find_repo_root(start=None):
    """Return the repository root — the folder that contains data/sp500_prices.csv."""  # docstring: what the function returns
    # Start from the given folder, or the current working folder, as a full absolute path
    here = Path(start or Path.cwd()).resolve()
    # Check this folder, then every folder above it in turn
    for candidate in [here, *here.parents]:
        # Does this candidate contain data/sp500_prices.csv?
        if (candidate / "data" / "sp500_prices.csv").exists():
            # Yes: this is the repository root
            return candidate
    # Searched all the way up with no match: stop with a clear instruction
    raise FileNotFoundError(
        "Could not find the repository root. In VS Code use File > Open Folder "  # message, part 1
        "and open the mtu4600-lab3-analytics folder itself, then re-run."  # part 2, glued onto part 1
    )


# Locate the project folder once and keep it as a Path object
REPO = find_repo_root()
# The folder holding the raw CSV files
DATA = REPO / "data"
# Apply the shared course chart style to every chart drawn from now on
plt.style.use(REPO / "fin4600.mplstyle")

# Show up to 30 columns when printing a DataFrame, instead of hiding the middle ones behind "..."
pd.set_option("display.max_columns", 30)
# Allow printed tables up to 120 characters wide before wrapping
pd.set_option("display.width", 120)

# Confirm where the repository was found
print("Repository root:", REPO)

## 1. Loading a file — and why `parse_dates` matters

First, deliberately load the file the *wrong* way, so you can see what goes
wrong. This is the single most common bug in student market-data code.

#### Background: `pd.read_csv` and dtypes

A CSV (comma-separated values) file is plain text. `pd.read_csv(path)` reads
it and returns a DataFrame, guessing each column's **dtype** from what it
sees:

| dtype | holds | example column here |
|---|---|---|
| `int64` | whole numbers | `volume` |
| `float64` | decimal numbers (and `NaN` for missing) | `open`, `close` |
| `str` / `object` | text | `ticker` |
| `datetime64` | points in time | `date`, but **only if you ask** |

pandas does not turn text into dates on its own, because a string like
`03/04/2015` is 3 April in London and 4 March in New York. Unless told, it
leaves the date column as text.

The cell also uses Python's `try` / `except`: run the code under `try`, and
if it raises the named error (`TypeError`), jump to the `except` block instead
of crashing. Here we *expect* the error and print it.

**What to look for:** the date column's dtype is `str` (text), and
subtracting two text dates raises
`TypeError: unsupported operand type(s) for -: 'str' and 'str'`.

In [ ]:
# Read the CSV with no options; pandas leaves the date column as plain text
wrong = pd.read_csv(DATA / "sp500_prices.csv")
# .dtype reports a column's data type; expect text ("str" or "object"), not a date type
print("dtype of the date column when we don't say anything:", wrong["date"].dtype)
# Explain why text dates are dangerous (these print lines are just the lesson)
print("\nSorting a text date is alphabetical, not chronological, which is fine")
# ISO dates (2015-03-09) sort correctly as text only because year comes first
print("for ISO dates but silently wrong for formats like 3/9/2015 vs 12/1/2014.")
# Lead-in to the demonstration below
print("\nAnd you cannot do arithmetic on it:")
# try: run the risky line; if it raises a TypeError, jump to the except block instead of crashing
try:
    # Latest minus earliest date: works for real dates, fails for text
    wrong["date"].max() - wrong["date"].min()
# Catch the expected error and keep it in a variable called exc
except TypeError as exc:
    # Print Python's own error message so you can see exactly what went wrong
    print("   TypeError:", exc)

Now load it properly. `parse_dates` converts the column to pandas'
`datetime64` type, which knows about calendars, ordering, and differences.


#### Background: `datetime64` and `Timedelta`

`parse_dates=["date"]` tells `read_csv` to convert the listed columns into
pandas' `datetime64` dtype as it reads them. Each value is then stored as a
count of time units since 1 January 1970 (the `[us]` in `datetime64[us]`
means microseconds), which is why dates now sort chronologically and support
arithmetic. Subtracting two datetimes gives a **`Timedelta`**, a length of
time such as `1825 days`. A datetime column also gains the `.dt` accessor
(`.dt.year`, `.dt.month`, `.dt.day_name()`), used later in this notebook.

**What to look for:** dtype `datetime64[us]`, a span of `1825 days`
(about five years), and the first five rows: Apple's prices from
8 February 2013.

In [ ]:
# Read the file again, this time converting the "date" column into real datetime values
prices = pd.read_csv(DATA / "sp500_prices.csv", parse_dates=["date"])

# The dtype is now datetime64, so dates know their calendar order
print("dtype of the date column now:", prices["date"].dtype)
# Date arithmetic now works: latest minus earliest gives a Timedelta (a length of time)
print("Span of the data:", prices["date"].max() - prices["date"].min())
# Show the first 5 rows; Jupyter displays the last line of a cell as a formatted table
prices.head()

## 2. First look: shape, dtypes, describe

Before any analysis, answer three questions: *How big is it? What types are
the columns? Do the numbers look plausible?*

**What the next cell does.**

- `prices.shape` is a tuple `(rows, columns)`. It is an *attribute*, so no
  brackets after it.
- `prices.info()` prints a one-screen summary: the index type
  (`RangeIndex`: plain row numbers 0 to 37,769), every column with its
  non-null count and dtype, and the memory used. A **non-null count** below
  the row count is the quickest way to spot missing values.

**What to look for:** `(37770, 7)`, 37,770 non-null values in every column
(no gaps in the raw file), four `float64` price columns, an `int64` volume
column, a text `ticker` column and a `datetime64` date column. 30 tickers
times 1,259 trading days is exactly 37,770 rows.

In [ ]:
# .shape is a (rows, columns) tuple: the size of the table
print("Rows and columns:", prices.shape)
# Print an empty line as a spacer
print()
# info() prints the index type, each column's non-null count and dtype, and memory use
prices.info()

`describe()` gives the five-number summary of every numeric column. Read it
sceptically — this is where you catch negative prices, zero volumes, and
prices that are off by a factor of 100 because someone stored cents.


#### Background: `describe()`

For every numeric (and datetime) column, `describe()` returns: `count`
(non-missing values), `mean`, `std` (standard deviation, the typical
distance from the mean), `min`, the 25th, 50th (median) and 75th
**percentiles**, and `max`. When the mean sits far above the median, the
column is *right-skewed*: a few very large values pull the mean up.

**What to look for:** the minimum close is about \$11 and the maximum about
\$1,188 (Alphabet, `GOOGL`), so nothing is negative or absurd. Mean volume
(about 16.7 million) is twice the median (about 8.3 million): trading volume
is right-skewed, as it always is. `1.667327e+07` is scientific notation for
$1.667 \times 10^7$. The `std` of the date column is `NaN` because a standard
deviation of dates is not defined.

In [ ]:
# Summary statistics (count, mean, std, min, quartiles, max) for every numeric and date column
prices.describe()

## 3. Selecting rows and columns

Three tools, and it is worth being clear about when to use each:

- `df["col"]` — one column, by name
- `df.loc[rows, cols]` — select by **label** (a ticker, a date, a column name)
- `df.iloc[rows, cols]` — select by **integer position** (row 0, row 5)

In practice, `.loc` with a boolean mask is what you will use 90% of the time.


#### Background: Series, DataFrame and the index

`prices["close"]` (single brackets, one name) returns a **Series**: one
column of values plus the index. `prices[["date", "ticker", "close"]]`
(a *list* of names inside the brackets, hence the double brackets) returns a
smaller **DataFrame**. The distinction matters because the two types have
different methods and print differently.

`.loc` selects by *label*: it looks up index labels and column names. `.iloc`
selects by *position*: `prices.iloc[0:5, 0:3]` is the first five rows and
first three columns whatever they are called. When the index is plain row
numbers, the label and the position happen to coincide, which hides bugs that
appear as soon as you filter or sort.

**What to look for:** `<class 'pandas.Series'>`, then a three-column table.

In [ ]:
# A single column is a Series
# type() reports which kind of object prices["close"] is: a pandas Series
print(type(prices["close"]))

# A list of columns is a DataFrame
# The inner [...] is a Python list of names; the result is a 3-column DataFrame; head(3) shows 3 rows
prices[["date", "ticker", "close"]].head(3)

**What the next cell does.** It filters rows with **boolean masks**: first all
of JPMorgan's rows, then JPMorgan's days that closed more than 3% below the
open.

#### Background: boolean masks

A comparison on a column, such as `prices["ticker"] == "JPM"`, is
*vectorised*: it returns a Series of `True` / `False`, one per row, aligned on
the same index. Passing that Series to `.loc[...]` keeps the rows marked
`True`. That is the whole idea, and it is how pandas filters.

To combine conditions use the element-wise operators `&` (and), `|` (or) and
`~` (not). The Python words `and` / `or` do not work on a whole column; they
raise `ValueError: The truth value of a Series is ambiguous`. Each condition
must sit in its own parentheses, because `&` and `|` bind *more tightly* than
`==` and `<`. Without the brackets, Python evaluates `"JPM" & prices["close"]`
first, which raises an error.

`prices["open"] * 0.97` is 97% of the open price, so
`close < open * 0.97` means "fell more than 3% from open to close".

**What to look for:** 1,259 JPM rows (every trading day) and 8 days with a
fall of more than 3%: the list includes 25 August 2015 (the China
devaluation sell-off), early 2016 (oil and growth fears) and 5 February 2018
(the "Volmageddon" volatility spike).

In [ ]:
# Boolean mask: one bank's rows only
# prices["ticker"] == "JPM" gives True/False per row; .loc keeps the True rows
jpm = prices.loc[prices["ticker"] == "JPM"]
# len() of the filtered DataFrame = number of JPM trading days
print("JPM rows:", len(jpm))

# Two conditions. Note the parentheses — & and | bind tighter than == and < in Python,
# so leaving the parentheses out makes Python evaluate "JPM" & prices["close"] first, which raises an error.
# Keep rows that are JPM AND whose close is more than 3% below that day's open
big_down_days = prices.loc[
    (prices["ticker"] == "JPM") & (prices["close"] < prices["open"] * 0.97)  # both masks must be True
]
# Count the qualifying days
print("JPM days that closed more than 3% below the open:", len(big_down_days))
# Show only four columns of those rows; the left-hand numbers are the original row labels from prices
big_down_days[["date", "open", "close", "volume"]]

**What the next cell does.** It selects all four banks at once. `.isin(list)`
returns `True` for every row whose value appears in the list, which is
cleaner than chaining four `==` tests with `|`. `nunique()` then counts the
distinct tickers kept.

**What to look for:** 5,036 rows across 4 tickers (4 × 1,259).

In [ ]:
# .isin() for a set of labels
# True where the ticker is any of the four banks; .loc keeps those rows
banks = prices.loc[prices["ticker"].isin(["JPM", "GS", "BAC", "WFC"])]
# Row count, and nunique() = how many distinct tickers survived the filter
print("Bank rows:", len(banks), "across", banks["ticker"].nunique(), "tickers")

### Exercise 1 (worked solution)

Select every row for `XOM` (Exxon) in calendar year 2015 where daily volume
was above 20 million shares. How many such days were there?

*Hint:* `prices["date"].dt.year` gives the year of a datetime column.

**How the solution works.** Build three separate boolean masks, give each a
readable name, and combine them with `&`. `.dt.year` extracts the year from
every date in the column at once (the `.dt` accessor only exists on datetime
columns, which is one more reason to parse dates). `20_000_000` is just
20,000,000; Python lets you put underscores in numbers for readability. The
solution uses new variable names so it does not disturb `prices`, and it adds
no columns to `prices` (the `year` column is created later, in section 4).

In [ ]:
# Mask 1: True on rows belonging to Exxon
is_xom = prices["ticker"] == "XOM"
# Mask 2: .dt.year pulls the calendar year out of each date; True for 2015 rows
in_2015 = prices["date"].dt.year == 2015
# Mask 3: True where more than 20 million shares traded (underscores are just digit separators)
heavy_volume = prices["volume"] > 20_000_000

# All three conditions must hold; & combines the masks row by row
xom_heavy_2015 = prices.loc[is_xom & in_2015 & heavy_volume]

# For context: how many trading days did XOM have in 2015 at all?
print("XOM trading days in 2015:            ", int((is_xom & in_2015).sum()))
# The answer to the exercise: number of rows passing all three masks
print("XOM days in 2015 with volume > 20m:  ", len(xom_heavy_2015))
# Group the qualifying days by calendar month and count them, to see when the heavy trading happened
print("\nHeavy-volume days by month of 2015:")
# .dt.month gives 1-12; value_counts() counts each month; sort_index() orders them January to December
print(xom_heavy_2015["date"].dt.month.value_counts().sort_index().to_string())
# Show the five busiest of those days, largest volume first
xom_heavy_2015.sort_values("volume", ascending=False)[["date", "open", "close", "volume"]].head()

**Discussion.** Exxon had **24** trading days in 2015 with volume above
20 million shares, out of 252 trading days that year (under 10%). They are
not spread evenly. The month-by-month count shows **8 in December** and
**5 in August**, and none at all in April, May, June or October.

- August's cluster includes 24–26 August 2015, the global sell-off after
  China devalued the yuan.
- The single busiest day, 20 March 2015 (about 36.1 million shares), and
  18 December 2015 were both third Fridays of a quarter-end month, when
  stock-index futures and options expire and index funds rebalance. Huge
  volume on those days is mechanical, not news.
- 2015 was also the second year of the oil price collapse, which kept
  trading in an oil major elevated.

Volume spikes are a signal in their own right: they mark days when the
market was either re-pricing the company or being forced to trade it.

## 4. Adding computed columns

Never write a Python loop over rows. pandas operations are *vectorised* —
they apply to the whole column at once, in compiled code. On this dataset the
difference is roughly a hundredfold, and on a real tick file it is the
difference between a report that runs and one that does not.


#### Background: vectorisation

A *vectorised* operation is written once for a whole column and executed by
NumPy in a tight compiled loop. `prices["high"] - prices["low"]` subtracts
37,770 pairs of numbers in one instruction; pandas lines the two columns up
by index label and returns a new Series. A Python `for` loop over rows does
the same work one row at a time, with interpreter overhead on every step.

Assigning to a new column name, `prices["new"] = ...`, adds that column to the
DataFrame (or overwrites it if the name exists).

**The three new columns:**

- `daily_range_pct` $= (\text{high} - \text{low}) / \text{close} \times 100$, the
  day's high-low range as a percentage of the close. It is a simple measure
  of *intraday volatility*: how far the price travelled during the day.
- `dollar_volume` $= \text{close} \times \text{volume}$, the approximate value
  traded in US dollars. It is a better liquidity measure than share volume,
  because 1 million shares of a \$1,000 stock is not the same as 1 million
  shares of a \$10 stock.
- `year`, via the `.dt.year` accessor, for grouping by year later.

**What to look for:** Apple's daily ranges of about 1.6–3.1% and dollar
volumes around \$6–11 billion a day (`1.073239e+10` is \$10.7 billion).

In [ ]:
# High minus low, divided by close, times 100: the day's trading range as a % of the closing price (whole column at once)
prices["daily_range_pct"] = (prices["high"] - prices["low"]) / prices["close"] * 100
# Price times shares traded = approximate US dollars traded that day (a liquidity measure)
prices["dollar_volume"] = prices["close"] * prices["volume"]
# The calendar year of each row, extracted from the datetime column with the .dt accessor
prices["year"] = prices["date"].dt.year

# Display a few columns, including the three new ones, for the first 5 rows
prices[["date", "ticker", "close", "daily_range_pct", "dollar_volume", "year"]].head()

## 5. `groupby` — the workhorse

"Split, apply, combine". Split the rows into groups, apply a function to each
group, combine the answers into one table. Every per-desk, per-sector, or
per-instrument report you will ever build is a `groupby`.


#### Background: how `groupby` works

`prices.groupby("ticker")["close"].mean()` runs in three steps:

1. **Split**: pandas looks at the `ticker` column and builds, for each
   distinct value, the list of row positions belonging to it (30 groups here).
2. **Apply**: it runs the function (`mean`) on the `close` values of each
   group separately.
3. **Combine**: it collects the 30 answers into a new Series whose **index is
   the group keys** (the tickers).

The built-in aggregations (`mean`, `sum`, `min`, `max`, `first`, `last`,
`size`, `count`, `std`, `nunique`) run in optimised compiled code.
`.sort_values(ascending=False)` then orders the result largest first, and
`.head(8)` keeps the top eight.

**What to look for:** Alphabet (`GOOGL`, about \$682) and Goldman Sachs
(`GS`, about \$188) top the list. Note what this does **not** tell you: a high
average *price* says nothing about return, risk or company size. It mostly
reflects how many shares a company has issued and whether it has split its
stock. Never compare stocks by price level.

In [ ]:
# One statistic per ticker
# Split rows by ticker, take the mean closing price of each group, then sort highest first
by_ticker = prices.groupby("ticker")["close"].mean().sort_values(ascending=False)
# Show the top 8 tickers (the index of this Series is the ticker)
by_ticker.head(8)

**What the next cell does.** It builds a per-ticker summary table with
several statistics at once, then adds a total-return column.

**Named aggregation.** Inside `.agg(...)`, each keyword argument has the form
`new_column_name=(source_column, function)`. The function can be a built-in
name in quotes (`"first"`, `"last"`, `"mean"`, `"size"`) or your own function.
`lambda s: s.mean() / 1e6` is a small unnamed function: given one group's
`dollar_volume` Series `s`, it returns the mean in millions (`1e6` is one
million). `"first"` and `"last"` return each group's first and last row, so
they rely on the file being sorted by date within each ticker, which it is.

**Total return.** $\text{total return} = \left(\frac{P_\text{last}}{P_\text{first}} - 1\right) \times 100\%$.
These closes are adjusted for stock splits (Apple's 7-for-1 split in 2014 is
why its 2013 price shows as about \$68) but not for dividends, so this is a
*price return*, slightly understating what a shareholder earned.

`.round(2)` rounds every number to 2 decimal places for display only.

**What to look for:** Boeing (`BA`) leads with about a 355% price return over
the five years, then UnitedHealth (about 291%) and Microsoft (about 225%).
Every ticker has `n_days` = 1,259.

In [ ]:
# Several statistics at once, with readable output column names
# Group by ticker; each keyword below is  new_name=(source column, aggregation)
summary = prices.groupby("ticker").agg(
    first_close=("close", "first"),  # closing price on the group's first row (earliest date)
    last_close=("close", "last"),  # closing price on the last row (latest date)
    avg_daily_range_pct=("daily_range_pct", "mean"),  # average intraday range, %
    avg_dollar_volume_mm=("dollar_volume", lambda s: s.mean() / 1e6),  # average $ traded per day, in millions
    n_days=("close", "size"),  # number of rows (trading days) in the group
)
# Total price return over the whole sample, in percent: last / first - 1, times 100
summary["total_return_pct"] = (
    summary["last_close"] / summary["first_close"] - 1  # growth factor minus 1 = return as a decimal
) * 100  # convert the decimal return to percent

# Sort best performer first, round to 2 dp for display, and show the top 10
summary.sort_values("total_return_pct", ascending=False).round(2).head(10)

**What the next cell does.** Groups by *two* keys, ticker and year, and
averages dollar volume within each (ticker, year) pair.

#### Background: the MultiIndex

When you group by two columns the result's index has two levels, a
**MultiIndex**: each row label is a pair such as `("AAPL", 2013)`. pandas
prints the outer level only when it changes, which is why `AAPL` appears
once. You can select with `by_ticker_year.loc["AAPL"]` (all of Apple's years)
or `by_ticker_year.loc[("AAPL", 2016)]` (one value). A MultiIndex is also
what `stack()` produces in section 6.

**What to look for:** Apple's average dollar volume falls from about \$6.4bn
a day in 2013 to about \$4.0bn in 2016, then rises again. The 2018 figure
covers only five weeks (to 7 February), which included a sharp sell-off, so
it is not comparable to a full year.

In [ ]:
# Group by two keys — the result has a MultiIndex
# Average daily dollar volume for every (ticker, year) pair; the index has two levels
by_ticker_year = prices.groupby(["ticker", "year"])["dollar_volume"].mean()
# First 6 rows: all of AAPL's years, 2013 to 2018
by_ticker_year.head(6)

### Exercise 2 (worked solution)

Which three tickers had the **highest average daily range** (`daily_range_pct`)
over the whole period? Does the answer match your intuition about which of
these businesses is riskiest?

**How the solution works.** It is a one-key `groupby` (split by ticker, average
`daily_range_pct`). `.nlargest(3)` returns the three biggest values with their
labels, a shortcut for `.sort_values(ascending=False).head(3)`; `.nsmallest(3)`
does the opposite, and gives the calmest names for contrast.

In [ ]:
# Average intraday range (%) for each ticker: split by ticker, mean of daily_range_pct
avg_range_by_ticker = prices.groupby("ticker")["daily_range_pct"].mean()

# The three largest averages, with their tickers as the index
print("Highest average daily range (%):")
# nlargest(3) = sort descending and keep the top 3; round to 3 dp; to_string() prints it without the dtype footer
print(avg_range_by_ticker.nlargest(3).round(3).to_string())
# For contrast, the three calmest stocks
print("\nLowest average daily range (%):")
# nsmallest(3) = the three smallest values
print(avg_range_by_ticker.nsmallest(3).round(3).to_string())
# The average across all 30 tickers, as a reference point
print(f"\nAverage across all 30 tickers: {avg_range_by_ticker.mean():.3f}%")

**Discussion.** The three widest average daily ranges belong to **Bank of
America (`BAC`, about 1.89%)**, **Goldman Sachs (`GS`, about 1.74%)** and
**Caterpillar (`CAT`, about 1.73%)**. The calmest are Coca-Cola (`KO`),
Procter & Gamble (`PG`) and McDonald's (`MCD`), all around 1.14–1.17%.

That broadly matches intuition. Banks are highly leveraged, so small changes
in the economy or interest rates move their equity a lot, and in 2013–2016
they were still working through post-crisis litigation and regulation.
Caterpillar is a textbook cyclical: its sales depend on mining, construction
and Chinese demand, all of which swung hard in this period. Consumer staples
sell soft drinks and toothpaste in good times and bad, so their prices move
little.

Two cautions. First, BAC traded at only \$11–\$31, and a stock with a low price
moves in larger *percentage* steps for the same one-cent tick, which nudges
its range upwards. Second, the high-low range measures **intraday** movement;
it is a useful volatility proxy but not the same thing as the volatility of
daily returns you will compute in Notebook 02.

## 6. Long vs wide: `pivot`

The file is in **long** format — one row per ticker per day. That is how
market data is stored and transmitted, because it handles instruments with
different histories cleanly.

Portfolio mathematics needs **wide** format — one row per day, one column per
ticker. Correlation matrices, covariance matrices and portfolio returns all
assume a wide matrix. Converting between the two is a daily task.


#### Background: `pivot` and the long/wide reshape

`prices.pivot(index="date", columns="ticker", values="close")` builds a new
table where:

- each distinct `date` becomes one **row** (the index),
- each distinct `ticker` becomes one **column**,
- each cell holds the `close` from the long row with that (date, ticker) pair.

`pivot` requires each (index, column) pair to appear at most once; a
duplicate raises an error. (If duplicates are legitimate, `pivot_table`
aggregates them, by default with the mean.) A pair that never appears becomes
`NaN`, which is how a wide table shows a stock that had not listed yet.

**What to look for:** long `(37770, 10)` (the 7 original columns plus the
3 added in section 4) becomes wide `(1259, 30)`: 1,259 trading days by 30
tickers. The columns appear in alphabetical order.

In [ ]:
# Reshape long -> wide: one row per date, one column per ticker, cells = closing price
close_wide = prices.pivot(index="date", columns="ticker", values="close")

# Long table: one row per (ticker, day)
print("Long format:", prices.shape)
# Wide table: one row per day, one column per ticker
print("Wide format:", close_wide.shape)
# .iloc by position: first 5 rows and first 6 columns
close_wide.iloc[:5, :6]

**What the next cell does.** Reverses the pivot.

- `.stack()` moves the column labels (tickers) down into the index. The
  result is a Series with a two-level MultiIndex `(date, ticker)`: long
  format again.
- `.rename("close")` names that Series, so it becomes a column called
  `close` in the next step.
- `.reset_index()` turns the index levels back into ordinary columns and
  gives the table a plain 0, 1, 2... row index.

Databases store data long because adding a new instrument then means adding
rows, not changing the table's structure.

**What to look for:** three columns `date`, `ticker`, `close`, now sorted by
date first and ticker second (AAPL, AMT, BA on 8 February 2013), the reverse
of the original file's order.

In [ ]:
# ...and back again, which you need when writing to a database
# stack(): ticker columns -> inner index level; rename(): name the values "close"; reset_index(): index levels -> columns
back_to_long = close_wide.stack().rename("close").reset_index()
# First 3 rows of the rebuilt long table
back_to_long.head(3)

## 7. `resample` — changing the time frequency

Daily marks, month-end reporting. `resample` only works on a datetime index,
which is why we pivoted first.

The rule strings you will use most: `"ME"` month-end, `"QE"` quarter-end,
`"YE"` year-end, `"W"` weekly.


#### Background: how `resample` works

`resample` is a `groupby` on time. Given a rule such as `"ME"`, pandas cuts
the datetime index into consecutive bins (calendar months here), assigns each
row to its bin, applies the aggregation to each bin, and labels each result
with the bin's **end date** (31 January, 28 February...). Like `groupby`, it
needs a method afterwards to say how to combine the rows in each bin:

- `.last()`: the last value in the month, which is the month-end mark;
- `.first()`, `.max()`, `.min()`, `.sum()`, `.mean()` do what they say.

A labelled month-end such as 2013-03-31 is a Sunday; the value is still the
close of the last *trading* day in March. Downsampling (daily to monthly) is
the common case; upsampling (monthly to daily) creates empty slots that must
be filled, which brings back the missing-data questions of section 9.

**What to look for:** 1,259 daily rows become 61 monthly rows (Feb 2013 to
Feb 2018 inclusive, both partial months).

In [ ]:
# Bin the daily rows into calendar months ("ME" = month-end) and keep each month's last close per ticker
monthly_close = close_wide.resample("ME").last()
# Number of daily rows going in
print("Daily observations:", len(close_wide))
# Number of monthly rows coming out
print("Monthly observations:", len(monthly_close))
# First 5 months and first 6 tickers, rounded to 2 dp for display
monthly_close.iloc[:5, :6].round(2)

**What the next cell does.** Builds monthly OHLCV (open, high, low, close,
volume) bars for Apple from its daily bars. Each column needs its own rule:

| monthly field | rule | why |
|---|---|---|
| `open` | `first` | the month opens at the first day's open |
| `high` | `max` | the highest price reached on any day |
| `low` | `min` | the lowest price reached on any day |
| `close` | `last` | the month closes at the last day's close |
| `volume` | `sum` | shares traded add up across days |

Averaging prices or taking the last volume would produce numbers no trader
would recognise. `.agg(...)` uses the same `name=(column, function)` syntax as
`groupby` in section 5. `set_index("date")` is needed first because
`resample` works on a datetime *index*.

**What to look for:** a monthly high always at least the monthly close, and
monthly volumes of about 1.6–2.7 billion shares.

In [ ]:
# Different columns often want different rules. Volume sums; price takes the last.
# Apple's rows only, with the date moved into the index so resample can use it
aapl_daily = prices.loc[prices["ticker"] == "AAPL"].set_index("date")
# Monthly bins; each keyword below is  new_name=(source column, rule)
aapl_monthly = aapl_daily.resample("ME").agg(
    open=("open", "first"),  # first open of the month
    high=("high", "max"),  # highest high of the month
    low=("low", "min"),  # lowest low of the month
    close=("close", "last"),  # last close of the month
    volume=("volume", "sum"),  # total shares traded in the month
)
# First 5 months, rounded to 2 dp
aapl_monthly.head().round(2)

## 8. Joining reference data: `merge`

Prices alone cannot answer "how did Financials do?". For that you need a
**security master** — the reference table that maps an instrument identifier
to its attributes. Every bank has one, and keeping it correct is a permanent
operational headache.


**What the next cell does.** Loads the security master and shows its first
rows. There are no dates to parse here, so a plain `read_csv` is fine.

**What to look for:** 100 rows (it covers more companies than the 30 in the
price file) with columns `ticker`, `company`, `sector`, `sub_industry`,
`headquarters`, `date_added` and `cik`. `ticker` is the **key**: the column
both tables share, used to match rows. Note that Airbnb (`ABNB`) was added to
the index in 2023, long after our price history ends, a first hint of the
time-mismatch problem discussed below. (`cik` is the SEC's company
identifier.)

In [ ]:
# Read the security master (ticker -> company name, GICS sector, ...) into a DataFrame
companies = pd.read_csv(DATA / "sp500_companies.csv")
# How many instruments the master covers
print("Security master rows:", len(companies))
# First 5 rows; the wide table wraps onto a second block because of the display width
companies.head()

### The join that quietly loses rows

An **inner** join keeps only rows that match on both sides. A **left** join
keeps every price row and leaves `NaN` where the master has no entry.

In production you almost always want a left join *plus an alert*, because a
silent inner join is how a position disappears from a risk report.


#### Background: `merge` and the four join types

`left_table.merge(right_table, on="ticker", how=...)` matches rows whose
`ticker` values are equal and glues the right table's columns onto the left
table's rows. It is the same operation as a SQL `JOIN`. The `how` argument
decides what happens to rows *without* a match:

| `how=` | keeps | unmatched rows |
|---|---|---|
| `"inner"` | only keys found in both tables | dropped silently |
| `"left"` | every row of the left table | right-hand columns filled with `NaN` |
| `"right"` | every row of the right table | left-hand columns filled with `NaN` |
| `"outer"` | every row of both | `NaN` on whichever side is missing |

Here the relationship is **many-to-one**: many price rows per ticker, one
master row per ticker, so a correct join never changes the number of price
rows. If the master had a *duplicate* ticker, every matching price row would
be copied twice and the row count would grow: the opposite failure. In
production code, `merge(..., validate="many_to_one")` makes pandas raise an
error if that ever happens, and `indicator=True` adds a column saying which
side each row came from.

`.isna()` returns `True` where a value is missing (`NaN`). `.unique()` returns
the distinct values. `list(unmatched) or "none"` prints the list, or the word
`none` when the list is empty (an empty list counts as false in Python).

**What to look for:** 37,770 rows before and after both joins, and no
unmatched tickers.

In [ ]:
# Inner join: keep only price rows whose ticker exists in the security master
inner = prices.merge(companies, on="ticker", how="inner")
# Left join: keep every price row; master columns become NaN where there is no match
left = prices.merge(companies, on="ticker", how="left")

# Row count going in
print("Price rows before the join:", len(prices))
# Row count after each join: identical counts mean nothing was lost or duplicated
print("After an inner join:       ", len(inner))
# Same check for the left join
print("After a left join:         ", len(left))

# Rows where sector is missing failed to match; take their ticker column and list the distinct values
unmatched = left.loc[left["sector"].isna(), "ticker"].unique()
# Print the list, or "none" if it is empty (an empty list counts as False, so "or" falls through)
print("\nTickers with no security-master entry:", list(unmatched) or "none")

Here every ticker matched. That is because the file was curated for this lab.
Reality is messier: the security master is a *current* index membership list,
while the price history covers 2013–2018, so a real join of the full files
would strand every company that has since been acquired or removed. Let's
simulate that, because recognising the symptom is the skill.


**What the next cell does.** Removes three tickers from a *copy* of the master
and repeats both joins. `~` means "not", so
`~companies["ticker"].isin([...])` keeps every company *except* GE, AT&T and
Duke Energy. The original `companies` table is untouched.

**What to look for:** the inner join loses 3,777 rows (3 tickers × 1,259 days)
with no error and no warning. The left join keeps all 37,770 rows and the
`NaN` sectors point straight at `DUK`, `GE` and `T`.

In [ ]:
# Pretend the security master lost three entries, as if those firms were acquired
# ~ flips the mask, so this keeps every company EXCEPT GE, T and DUK (a new table; companies is unchanged)
damaged = companies.loc[~companies["ticker"].isin(["GE", "T", "DUK"])]

# Inner join against the damaged master: unmatched price rows vanish
risky_inner = prices.merge(damaged, on="ticker", how="inner")
# Left join against the damaged master: unmatched rows stay, with NaN in the master's columns
safe_left = prices.merge(damaged, on="ticker", how="left")

# How many rows the inner join threw away; :, adds a thousands separator
print(f"Inner join silently dropped {len(prices) - len(risky_inner):,} rows.")
# Heading for the left-join diagnosis
print("Left join kept them all and flagged the problem:")
# Tickers whose sector is NaN after the left join = the ones missing from the master, sorted alphabetically
print("   unmatched tickers:", sorted(safe_left.loc[safe_left['sector'].isna(), 'ticker'].unique()))

**Rule of thumb:** after every join, compare the row count to what you
expected. A join that changes the row count unexpectedly is a bug until
proven otherwise.


**What the next cell does.** Uses the `left` join (with the full, undamaged
master) to answer the sector question: group by `sector`, count distinct
tickers with `"nunique"`, and average the range and dollar volume.

Wrapping the whole expression in outer parentheses lets you put one method
per line (`.groupby`, `.agg`, `.sort_values`). This *method chaining* style
reads top to bottom like a recipe and is standard in pandas code.

**What to look for:** Financials (about 1.66%) and Information Technology
(about 1.65%) have the widest average daily ranges; Consumer Staples (about
1.19%) the narrowest. The ticker counts add up to 30. Several sectors hold
only one to three stocks, so these are small-sample averages.

In [ ]:
# Now the sector question is answerable
# Outer brackets let the method chain run over several lines
sector_stats = (
    left.groupby("sector")  # split the joined price rows by GICS sector
    .agg(  # several named statistics per sector
        tickers=("ticker", "nunique"),  # how many distinct stocks are in the sector
        avg_daily_range_pct=("daily_range_pct", "mean"),  # average intraday range, %
        avg_dollar_volume_mm=("dollar_volume", lambda s: s.mean() / 1e6),  # average $ traded per day, millions
    )
    .sort_values("avg_daily_range_pct", ascending=False)  # most volatile sector first
)
# Display rounded to 2 dp
sector_stats.round(2)

**What the next cell does.** Draws a horizontal bar chart of each sector's
average daily range.

- `plt.subplots(figsize=(9, 4))` creates a figure 9 inches wide by 4 tall and
  one set of axes (`ax`), the plotting area.
- `.sort_values()` (ascending) puts the smallest bar first. `barh` draws the
  first item at the *bottom*, so the widest-range sector ends up at the top.
- `ax.barh(labels, lengths, ...)` draws horizontal bars; `height=0.65` is
  the bar thickness, `color` a hex colour code.
- `ax.grid(axis="x")` keeps vertical gridlines to read lengths against;
  the horizontal ones are switched off as clutter.
- The `for` loop uses `enumerate`, which yields `(position, value)` pairs:
  bar 0, 1, 2... and its length. `ax.text(x, y, text)` writes each value just
  past the end of its bar (`v + 0.02`), formatted to two decimals with a `%`.

**What to look for:** Financials and Information Technology at the top,
Consumer Staples clearly at the bottom, and most sectors bunched between
about 1.4% and 1.6%.

In [ ]:
# A 9 x 4 inch figure with one set of axes
fig, ax = plt.subplots(figsize=(9, 4))
# The sector ranges, smallest first, so the largest bar is drawn at the top of a horizontal chart
plot_data = sector_stats["avg_daily_range_pct"].sort_values()
# Horizontal bars: y positions labelled by sector name, bar length = average range
ax.barh(plot_data.index, plot_data.values, color="#2a78d6", height=0.65)
# Chart title
ax.set_title("Average daily high–low range by GICS sector, 2013–2018")
# x-axis label with units
ax.set_xlabel("Daily range as % of closing price")
# Vertical gridlines help read bar lengths
ax.grid(axis="x")
# Horizontal gridlines would just add clutter
ax.grid(axis="y", visible=False)
# enumerate gives (bar number y, bar length v) for each bar
for y, v in enumerate(plot_data.values):
    # Write the value, e.g. "1.66%", just to the right of the bar end, vertically centred, small grey text
    ax.text(v + 0.02, y, f"{v:.2f}%", va="center", fontsize=9, color="#52514e")
# Render the chart
plt.show()

## 9. Missing data

Markets close. Stocks halt. Feeds arrive late. Companies delist. Your data
will have holes, and how you fill them changes your answer.


**What the next cell does.** Compares three calendars over the sample period
using `pd.date_range(start, end, freq=...)`, which generates a sequence of
dates:

- `freq="D"`: every calendar day;
- `freq="B"`: every business day (Monday to Friday). pandas' `"B"` knows
  nothing about holidays;
- the index of `close_wide`: the days the market actually traded.

**What to look for:** 1,826 calendar days, 1,304 business days, 1,259 trading
days, so **45** weekday market holidays (about nine a year, which matches the
NYSE holiday calendar).

In [ ]:
# The trading calendar is not the calendar calendar
# Every calendar day from the first to the last date in the data
all_days = pd.date_range(close_wide.index.min(), close_wide.index.max(), freq="D")
# Every Monday-to-Friday in the same range ("B" = business day; it does not know about holidays)
business_days = pd.date_range(close_wide.index.min(), close_wide.index.max(), freq="B")

# Count of calendar days
print("Calendar days in the period: ", len(all_days))
# Count of weekdays
print("Business days:               ", len(business_days))
# Count of days the exchange actually traded (rows in the wide price table)
print("Actual trading days in data: ", len(close_wide))
# Weekdays with no trading = exchange holidays
print("Difference (market holidays):", len(business_days) - len(close_wide))

**What the next cell does.** Re-labels `close_wide` onto the full business-day
calendar and finds the gaps.

#### Background: `reindex` and `NaN`

`df.reindex(new_index)` returns a table with exactly the rows in
`new_index`, in that order. Rows that existed keep their values; new labels
get `NaN` ("not a number"), pandas' marker for a missing value. `NaN` is a
special floating-point value, so a column containing it must be `float`.

`.isna()` turns the table into `True` / `False` (missing or not). The first
`.sum()` counts `True`s per column (`True` counts as 1); the second adds up
the 30 column totals. `d.day_name()` gives the weekday's name.

**What to look for:** 1,350 missing values (45 holidays × 30 tickers) and the
familiar US holidays: Presidents' Day, Good Friday, Memorial Day, 4 July,
Labor Day, Thanksgiving, Christmas and New Year's Day.

In [ ]:
# Reindexing onto every business day exposes the holidays as NaN
# Rows for every weekday; days not in close_wide (holidays) get NaN for every ticker
reindexed = close_wide.reindex(business_days)
# isna() -> True/False table; first sum() counts per column, second sum() totals the columns
print("Missing values introduced:", int(reindexed.isna().sum().sum()))

# The index labels (dates) where AAPL has no price = market holidays
holidays = reindexed.index[reindexed["AAPL"].isna()]
# Heading line
print("\nFirst eight market holidays in the sample:")
# Loop over the first eight holiday dates
for d in holidays[:8]:
    # .date() drops the time part; .day_name() gives "Monday", "Thursday", ...
    print("   ", d.date(), d.day_name())

Three ways to deal with a hole, and they are not interchangeable:

| Method | What it does | When it is right |
|---|---|---|
| `.dropna()` | delete the row | the observation genuinely does not exist |
| `.ffill()` | carry the last price forward | a market holiday — the position still exists, the price just did not change |
| `.interpolate()` | straight-line between neighbours | almost never for prices; it invents a price that never traded |

**Forward fill is right for a holiday. Interpolation is not**, because it uses
tomorrow's price to fill in today — information you would not have had. That
is *look-ahead bias*, and it is the defect that makes most amateur backtests
worthless. Notebook 03 returns to this.

### A real example: Chevron, Thanksgiving 2014

US markets were closed on Thursday 27 November 2014. That same day, OPEC met
in Vienna and declined to cut production. When the market reopened on the
Friday, oil and the energy sector fell hard.

Look at what each filling method puts in the Thursday hole.


#### Background: how each filling method works

- **`dropna()`** removes every row (or, with `axis=1`, every column) that
  contains a `NaN`. Simple, but on a wide table one missing stock deletes the
  whole day for every other stock.
- **`ffill()`** (forward fill) walks down the column and replaces each `NaN`
  with the most recent non-missing value *above* it. It only ever looks
  backwards in time, so it never uses future information. Its mirror image,
  `bfill()` (back fill), copies the next value upwards and therefore *does*
  look ahead.
- **`interpolate()`** (default `method="linear"`) draws a straight line
  between the last value before the gap and the first value after it, and
  reads the missing points off that line. For a one-day gap between $P_{t-1}$
  and $P_{t+1}$ it fills $(P_{t-1} + P_{t+1}) / 2$. Because it needs
  $P_{t+1}$, it looks ahead.

**What the next cell does.** Builds a small DataFrame from a Python
dictionary: each key (`"raw"`, `"ffill"`, `"interpolate"`) becomes a column
name and each value a Series, aligned on the shared date index. It then
slices a date range with `.loc["2014-11-24":"2014-12-01"]`; on a datetime
index, `.loc` accepts date strings and, unlike Python's usual slicing,
*includes* both ends.

**What to look for:** on 27 November the raw column is `NaN`, forward fill
shows 115.11 and interpolation shows 111.99, which is exactly
$(115.11 + 108.87)/2$, the midpoint of Wednesday's and Friday's closes.

In [ ]:
# Put three versions of Chevron's price side by side; dict keys become column names
comparison = pd.DataFrame(
    {  # a dict: column name -> Series
        "raw": reindexed["CVX"],  # with the holiday NaNs left in
        "ffill": reindexed["CVX"].ffill(),  # each NaN replaced by the last traded price before it
        "interpolate": reindexed["CVX"].interpolate(),  # each NaN replaced by a straight line to the next price
    }
)
# Date-range slice on the datetime index (both ends included), rounded to 2 dp
comparison.loc["2014-11-24":"2014-12-01"].round(2)

Forward fill puts **115.11** in the Thursday hole — the last price that
actually traded, which is what a position would have been marked at.

Interpolation puts roughly **112** there, a number that is part-way to
Friday's post-OPEC price. Nobody could have known that on Thursday. If you
then computed a Thursday-to-Friday return from the interpolated series, your
model would appear to have anticipated the OPEC announcement — and your
backtest would show skill that does not exist.

### Exercise 3 (worked solution)

Count how many trading days each ticker actually has in `close_wide`. Are
they all the same? What would a ticker with fewer days tell you about that
company's history?

*Hint:* `close_wide.notna().sum()`

**How the solution works.** `notna()` is the opposite of `isna()`: `True`
wherever a price exists. Summing down each column counts the `True`s, giving
one number per ticker. `value_counts()` then counts how many tickers share
each day-count; a single line in its output means every ticker has the same
history length.

In [ ]:
# True where a price exists; sum() down each column counts them -> one number per ticker
days_per_ticker = close_wide.notna().sum()

# How many tickers have each day-count (one line of output = all identical)
print("Tickers grouped by number of trading days:")
# value_counts() tallies the distinct values in the Series
print(days_per_ticker.value_counts().to_string())
# nunique() == 1 means there is only one distinct day-count across all tickers
print("\nAll tickers have the same number of days:", days_per_ticker.nunique() == 1)
# Fewest and most days, as a cross-check
print("Fewest days:", days_per_ticker.min(), "  Most days:", days_per_ticker.max())

**Discussion.** Every one of the 30 tickers has exactly **1,259** trading
days, the full length of the sample, so the wide table has no gaps at all.

A ticker with fewer days would be telling you something about the company:
it **listed later** (an IPO or spin-off after February 2013, so its column
starts with `NaN`s), it **stopped trading** (acquired, merged, went bankrupt
or was delisted, so its column ends early), it was **halted** for a period,
or it changed ticker symbol and the data vendor split its history in two.
Occasionally it is simply a data-feed gap that needs fixing at source.

The fact that all 30 are complete is itself a warning. These stocks were
chosen because they are large, well-known companies that existed throughout
the period. A dataset built only from today's survivors leaves out the firms
that failed or were taken over, which tends to make historical returns look
better than they really were. That is **survivorship bias**, and the uniform
1,259 count is what it looks like in the data.

## 10. Saving your work

Save the wide price matrix — Notebook 02 picks it up from here.


**What the next cell does.** Builds the path `data/processed`, creates that
folder if it is missing (`mkdir(exist_ok=True)` does nothing if the folder
already exists, instead of raising an error), and writes `close_wide` to CSV
with `to_csv`. The date index is written as the first column, so Notebook 02
can read it back with `index_col` and `parse_dates`.

Saving an intermediate result is a normal step in a data pipeline: the later
notebook does not have to repeat the loading and reshaping.

**What to look for:** the path written and `1259 rows x 30 columns`.

In [ ]:
# Path of the folder for cleaned/derived data
processed = REPO / "data" / "processed"
# Create the folder; exist_ok=True means "no error if it is already there"
processed.mkdir(exist_ok=True)

# Write the wide price table to CSV; the date index becomes the first column
close_wide.to_csv(processed / "close_wide.csv")
# Confirm the file location
print("Wrote", processed / "close_wide.csv")
# .shape[0] = rows, .shape[1] = columns
print(f"{close_wide.shape[0]} rows x {close_wide.shape[1]} columns")

---

## Recap

1. Parse dates on load, or nothing downstream works.
2. `describe()` before you model — it catches bad data cheaply.
3. `groupby` is split-apply-combine; it answers nearly every "by desk / by
   sector / by instrument" question.
4. Market data is stored long and analysed wide. `pivot` converts.
5. Check the row count after every `merge`.
6. Forward-fill holidays. Do not interpolate prices.

## Your turn

1. Make sure you have run every cell from top to bottom without errors
   (**Run All** in the notebook toolbar is fine).
2. Read the comments in each code cell, the background notes, and the worked
   solutions and discussions for the three exercises.
3. Complete the **Canvas comprehension quiz for Notebook 01**. It checks that
   you understand what the code does, how the pandas methods work, and what
   the results mean for a finance professional.

Then commit and push your run notebook:

```
git add notebooks/01_pandas_fundamentals.ipynb
git commit -m "Completed notebook 01"
git push
```

Next: `02_returns_and_risk.ipynb`